# Template application

In [ ]:
entity_template_file = 'entity.templ.html'
attribute_template_file = 'attribute.templ.html'

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
from lxml import etree
import IPython

# valid xml namespaces and schema for Confluence 6 storage format
# See 
xml_namespaces=[
    'xmlns="http://www.w3.org/1999/xhtml"',
    'xmlns:ac="http://www.atlassian.com/schema/confluence/4/ac/"',
    'xmlns:ri="http://www.atlassian.com/schema/confluence/4/ri/"',
    'xmlns:acxhtml="http://www.atlassian.com/schema/confluence/4/"'
]

def encapsulate_storage_format(xml):
    return '<?xml version="1.0"?><root doc="container to properly encapsulate xml" {} >\n{}\n</root>'.format(
            ' '.join(xml_namespaces), xml)

def beautify_xml(flat_xml):
    try:
        encapsulated = encapsulate_storage_format(flat_xml)
        dom = xml.dom.minidom.parseString(encapsulated)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        log.error('Expat error {}'.format(e))
        raise Exception(e)
    
def validate_storage_format(xml_in_storage_format):
        '''Validate if input xml conforms to Confluence storage format specifications'''
        parser = etree.XMLParser(dtd_validation=False)
        try:
            etree.fromstring(encapsulate_storage_format(xml_in_storage_format), parser)
            return None
        except xml.parsers.expat.ExpatError as e:
            m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
            message = 'Malformed xml ' + flat_xml[:50] + ' ...'
            if m:
                lines = wrapped.splitlines()
                messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
            else:
                message = message + flat_xml[:50] + ' ...'
            return message

beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('all fine!')

# Publisher as in Publisher.py

In [ ]:
from xml.sax.saxutils import escape
from datetime import datetime
import logging

class Publisher:
    '''
    The publisher contains mapping information of IM elements.
    And helper methods to translate content from IM json to Confluence
    '''

    log = logging.getLogger(__name__)

    def __init__(self, config, data, confluence, space_key, root_page_id, language='de'):
        self.config = config
        self.json_data = data
        self.confluence = confluence
        self.space_key = space_key
        self.root_page_id = root_page_id
        self.language = language

        # dictionary with key = element_key ('E233322', 'A132452', ...)
        self.content_map = {}
        stamp_now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    def translate(self, field):
        if field and field.get(self.language):
            return escape(field[self.language])
        return ''

    def page_title(self, key: str):
        '''Returns the page title of an element. This will be used to reference elements'''
        page = self.content_map[key]
        return page['title']

    def scan_current_content(self):
        '''Scan current content below page-root and fills the content_map accordingly'''
        None   # Nothing found

    def register_page(self, key: str, page_id: str, title: str):
        page = self.content_map.get(key)
        if page:
            self.log.warning('Element {} entry {} will be overwritten'.format(key, page))
        page = {}
        page['pageid'] = page_id
        page['title'] = title
        self.content_map[key] = page

    def page_for_key(self, key: str):
        '''Returns the page object of an element or None if there is no page yet
        A page object is a dictionary containing 'pageid' and 'name'
        '''
        return self.content_map.get(key)

    def stub(self, title: str, parent_page_id):
        '''Create a stub page to obtain the page id for the title'''
        if self.confluence.page_exists(self.space_key, title):
            key = self.confluence.get_page_id(self.space_key, title)
            if key:
                return { 'id': str(key) }
            assert false, 'illegal condition, page exists but unable to retreive the id'
        create_result = self.confluence.create_page(self.space_key, title=title, parent_id=parent_page_id, body=('generated stub'))
        return create_result

    def update_page(self, key: str, body: str):
        meta = self.page_for_key(key)
        self.confluence.update_page(meta['pageid'], meta['title'], body)

    def relation_self(self, entity_key: str, relation_key: str):
        '''Returns the local end of the relation_key attached to enitity_key'''
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['from-to']
        else:
            return relation['to-from']

    def relation_other(self, entity_key: str, relation_key: str):
        '''Returns the remote end of the relation_key'''
        relation = self.json_data['relations'][relation_key]
        if relation['from-to']['enti'] == entity_key:
            return relation['to-from']
        else:
            return relation['from-to']
    



# Load datasource

In [ ]:
import json

data = None
with open('testdata/IM-sample.json', 'r') as source:
     data = json.load(source)

entities = data['entities']
# Print some information on what was loaded
print('Model "{}" contains {} entities:'.format(data['model']['name'], len(entities)))

In [ ]:
publisher = Publisher(None, data, None, None, None, language='de')
list(map(lambda e: (e, publisher.translate(entities[e]['name'])), data['entities']))[:5]

## Load page mappings
This will be done during the scan phase in production mode.

In [ ]:
elements = [ 'entities', 'attributes', 'relations']

for element_type in elements:
    for key in data[element_type]:
        entitiy = data[element_type][key]
        name = entitiy.get('name')
        if type(name) is dict:
            title = publisher.translate(entitiy['name']).strip()
            publisher.register_page(key, -1, title)
        else: # relations have no name
            title = key # HACK: better use 'Entity Name - Entity Name'
            if type(name) is str:
                title = name
            publisher.register_page(key, -1, title)

In [ ]:
publisher.relation_self('ENTI12753', 'RELA13060')

In [ ]:
publisher.relation_other('ENTI12701', 'RELA13060')

# Sandbox for entity template

In [ ]:
test_entity_name = 'ENTI12701' # Attribut
test_entity_name = 'ENTI12753' # Entität

test_entity = data['entities'][test_entity_name]
test_entity_title = publisher.translate(test_entity['name'])
log.warning('Working with entity {} Name: "{}"'.format(test_entity_name, test_entity_title))
test_entity

In [ ]:
entity_template = None
with open('./templates/' + entity_template_file, 'r') as f:
    entity_template = f.read()

assert entity_template

IPython.display.Code(entity_template)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)

entity_template = env.get_template(entity_template_file)
rendered_entity_template = entity_template.render(key=test_entity_name, item=test_entity, util=publisher, data=data)
entity_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_entity_template) # strip comment lines
IPython.display.Code(entity_content_xml)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(test_entity_title, entity_content_xml))

In [ ]:
validate_storage_format(entity_content_xml)

# Sandbox for attribute template

In [ ]:
first_attribute_name = 'ATTR12612'
test_attribute = data['attributes'][first_attribute_name]
assert test_attribute

test_attribute_title = publisher.translate(test_attribute['name'])

log.warning('Working with attribute ' + first_attribute_name + ". Name: " + test_attribute['name']['de'])
test_attribute

In [ ]:
attribute_template = env.get_template(attribute_template_file)
rendered_attribute_template = attribute_template.render(key=first_attribute_name, item=test_attribute, util=publisher)
attribute_content_xml = re.sub('<!--.+?->(\n+)*', '', rendered_attribute_template) # strip comment lines
IPython.display.Code(attribute_content_xml)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format(test_attribute_title, attribute_content_xml))

# Publish to test space to verify result in Confluence

In [ ]:
import yaml
import copy

with open('config.yaml') as f:
    config = yaml.safe_load(f)

space_key = config['confluence']['space']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])

In [ ]:
entity_result = confluence.update_or_create(root_page_id, '{} - Test entity'.format(test_entity_title), entity_content_xml)
attribute_result = confluence.update_or_create(root_page_id, '{} - Test attribute'.format(test_attribute_title), attribute_content_xml)

In [ ]:
e_page = entity_result['_links']['webui']
e_uri = '{}{}'.format(config['confluence']['apiurl'], e_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    e_uri, test_entity_title))

In [ ]:
a_page = attribute_result['_links']['webui']
a_uri = '{}{}'.format(config['confluence']['apiurl'], a_page)
IPython.display.HTML('Link zur Confluence-Page: <a href="{}" target="_blank">{}</a>'.format(
    a_uri, test_attribute_title))